In [5]:
import os
import glob
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.layers import BatchNormalization, Dropout, Conv2D, MaxPooling2D, Dense, Flatten, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.metrics import classification_report
import xml.etree.ElementTree as ET

# Function to parse annotations from RML files
def parse_annotations(rml_file_path):
    annotations = []
    try:
        tree = ET.parse(rml_file_path)
        root = tree.getroot()
        for event in root.findall(".//ns0:Event", namespaces={"ns0": "http://www.respironics.com/PatientStudy.xsd"}):
            family = event.attrib.get("Family", "").strip()
            type_ = event.attrib.get("Type", "").strip()
            start = event.attrib.get("Start", "0").strip()
            duration = event.attrib.get("Duration", "0").strip()

            try:
                start = float(start)
                duration = float(duration)
            except ValueError:
                print(f"Invalid Start or Duration in annotation: Start={start}, Duration={duration}")
                continue

            if family == "Respiratory":
                annotations.append({"family": family, "type": type_, "start": start, "duration": duration})
    except Exception as e:
        print(f"Error parsing {rml_file_path}: {e}")
    return annotations

# Function to extract statistical features from audio
def extract_statistical_features(wav_file_path, annotations, sr=22050):
    features = []
    labels = []
    try:
        audio, _ = librosa.load(wav_file_path, sr=sr)
        audio_length = len(audio)
        print(f"Audio length (samples): {audio_length}")

        for annotation in annotations:
            start = annotation.get("start", 0)
            duration = annotation.get("duration", 0)

            start_sample = max(0, int(start * sr))
            end_sample = min(audio_length, int(start_sample + duration * sr))

            if end_sample > start_sample:
                segment = audio[start_sample:end_sample]

                # Extract statistical features
                zcr = np.mean(librosa.feature.zero_crossing_rate(y=segment))
                rmse = np.mean(librosa.feature.rms(y=segment))
                spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=segment, sr=sr))
                spectral_bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=segment, sr=sr))
                spectral_flatness = np.mean(librosa.feature.spectral_flatness(y=segment))
                spectral_rolloff = np.mean(librosa.feature.spectral_rolloff(y=segment, sr=sr))

                # Combine features
                feature_vector = [zcr, rmse, spectral_centroid, spectral_bandwidth, spectral_flatness, spectral_rolloff]
                features.append(feature_vector)
                labels.append(annotation.get("type", "Unknown"))
            else:
                print(f"Skipping annotation with invalid segment: Start={start_sample}, End={end_sample}")
    except Exception as e:
        print(f"Error processing file {wav_file_path}: {e}")
    return np.array(features), labels

# Build CNN model
def build_cnn(input_shape, num_classes):
    model = tf.keras.Sequential([
        Input(shape=input_shape),
        Conv2D(32, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.3),

        Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.3),

        Conv2D(128, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.3),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    return model

# Main function to process folders and train CNN
def train_cnn_model(wav_folder_path, rml_folder_path, sr=22050, n_mels=128, target_frames=100):
    all_features = []
    all_labels = []

    wav_files = glob.glob(os.path.join(wav_folder_path, "*.wav"))
    rml_files = glob.glob(os.path.join(rml_folder_path, "*.rml"))

    for wav_file_path in wav_files:
        wav_base = os.path.basename(wav_file_path).split("_")[0]
        matching_rml_file = next((rml for rml in rml_files if wav_base in os.path.basename(rml)), None)
        if not matching_rml_file:
            print(f"No matching RML file for {wav_file_path}")
            continue

        annotations = parse_annotations(matching_rml_file)
        features, labels = extract_statistical_features(wav_file_path, annotations, sr)
        if features.size > 0:
            all_features.extend(features)
            all_labels.extend(labels)

    if not all_features or not all_labels:
        print("No data available for training.")
        return

    all_features = np.array(all_features)
    all_features = all_features[..., np.newaxis]
    label_encoder = LabelEncoder()
    all_labels_encoded = label_encoder.fit_transform(all_labels)

    X_train, X_test, y_train, y_test = train_test_split(all_features, all_labels_encoded, test_size=0.2, random_state=42)

    model = build_cnn((all_features.shape[1], 1), len(label_encoder.classes_))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, verbose=1, restore_best_weights=True)

    model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=30, batch_size=32, callbacks=[lr_scheduler, early_stopping])

    loss, accuracy = model.evaluate(X_test, y_test)
    print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

# Specify folder paths and execute
wav_folder_path = "/Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_wav/Mic"
rml_folder_path = "/Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_label"

# Execute
train_cnn_model(wav_folder_path, rml_folder_path)


Audio length (samples): 400141350
Audio length (samples): 368830350
Audio length (samples): 348147450
Audio length (samples): 416237850
Audio length (samples): 322745850
Audio length (samples): 322216650
Audio length (samples): 417671100
Audio length (samples): 286363350
Audio length (samples): 280850850
Audio length (samples): 358885800
Audio length (samples): 336681450
Audio length (samples): 383251050
Audio length (samples): 232384950
Audio length (samples): 424219950
Audio length (samples): 262725750
Audio length (samples): 334961550
Audio length (samples): 319306050
Audio length (samples): 376173000
Audio length (samples): 305833500
Audio length (samples): 315469350
Audio length (samples): 358731450
Audio length (samples): 299813850
Audio length (samples): 356548500
Audio length (samples): 283673250
Audio length (samples): 503930700
Audio length (samples): 352888200
Audio length (samples): 364199850
Audio length (samples): 360032400
Audio length (samples): 391409550
Audio length (

ValueError: Input 0 of layer "conv2d" is incompatible with the layer: expected min_ndim=4, found ndim=3. Full shape received: (None, 6, 1)

In [12]:
import os
import glob
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# CNN model definition
def build_cnn(input_shape, num_classes):
    """
    Build a CNN model with adjusted kernel sizes, padding, and dropout for better generalization.
    """
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv2D(32, (2, 2), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2), padding='same'),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Conv2D(64, (2, 2), activation='relu', padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2), padding='same'),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Training function
def train_cnn_model(wav_folder_path, rml_folder_path, sr=22050, batch_size=32, epochs=30):
    all_features = []
    all_labels = []

    # Collect features and labels
    wav_files = glob.glob(os.path.join(wav_folder_path, "*.wav"))
    rml_files = glob.glob(os.path.join(rml_folder_path, "*.rml"))

    for wav_file_path in wav_files:
        wav_base = os.path.basename(wav_file_path).split("_")[0]
        matching_rml_file = next((rml for rml in rml_files if wav_base in os.path.basename(rml)), None)
        if not matching_rml_file:
            print(f"No matching RML file for {wav_file_path}")
            continue

        annotations = parse_annotations(matching_rml_file)
        features, labels = extract_statistical_features(wav_file_path, annotations, sr)
        if features.size > 0:
            all_features.extend(features)
            all_labels.extend(labels)

    if not all_features or not all_labels:
        print("No data available for training.")
        return

    # Convert features and reshape for Conv2D
    all_features = np.array(all_features, dtype=np.float32)
    all_features = all_features[..., np.newaxis, np.newaxis]  # Add two channel dimensions

    # Encode labels
    label_encoder = LabelEncoder()
    all_labels_encoded = label_encoder.fit_transform(all_labels)

    # Compute class weights to handle class imbalance
    class_weights = compute_class_weight(
        'balanced',
        classes=np.unique(all_labels_encoded),
        y=all_labels_encoded
    )
    class_weights = dict(enumerate(class_weights))

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(all_features, all_labels_encoded, test_size=0.2, random_state=42)

    # Build and compile the model
    model = build_cnn((all_features.shape[1], all_features.shape[2], 1), len(label_encoder.classes_))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # Callbacks for better training
    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    early_stopping = EarlyStopping(monitor='val_loss', patience=5, verbose=1, restore_best_weights=True)

    # Train the model
    model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[lr_scheduler, early_stopping],
        class_weight=class_weights  # Apply class weights
    )

    # Evaluate the model
    loss, accuracy = model.evaluate(X_test, y_test)
    print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

    # Generate predictions and classification report
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

# Specify the folder paths
# Specify folder paths and execute
wav_folder_path = "/Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_wav/Mic"
rml_folder_path = "/Users/terlan/Library/CloudStorage/OneDrive-uni-mannheim.de/cleaned_label"

# Execute the training
train_cnn_model(wav_folder_path, rml_folder_path)


Audio length (samples): 400141350
Audio length (samples): 368830350
Audio length (samples): 348147450
Audio length (samples): 416237850
Audio length (samples): 322745850
Audio length (samples): 322216650
Audio length (samples): 417671100
Audio length (samples): 286363350
Audio length (samples): 280850850
Audio length (samples): 358885800
Audio length (samples): 336681450
Audio length (samples): 383251050
Audio length (samples): 232384950
Audio length (samples): 424219950
Audio length (samples): 262725750
Audio length (samples): 334961550
Audio length (samples): 319306050
Audio length (samples): 376173000
Audio length (samples): 305833500
Audio length (samples): 315469350
Audio length (samples): 358731450
Audio length (samples): 299813850
Audio length (samples): 356548500
Audio length (samples): 283673250
Audio length (samples): 503930700
Audio length (samples): 352888200
Audio length (samples): 364199850
Audio length (samples): 360032400
Audio length (samples): 391409550
Audio length (

/var/folders/36/xfvf6zw53bd2ht35p52k28qh0000gn/T/ipykernel_15753/3094099721.py:44: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, _ = librosa.load(wav_file_path, sr=sr)
/Users/terlan/anaconda3/lib/python3.10/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Audio length (samples): 326295900
Epoch 1/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 23s 20ms/step - accuracy: 0.2528 - loss: 2.4250 - val_accuracy: 0.4942 - val_loss: 1.3512 - learning_rate: 0.0010
Epoch 2/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.2423 - loss: 3.4682 - val_accuracy: 0.4944 - val_loss: 1.3127 - learning_rate: 0.0010
Epoch 3/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 21s 20ms/step - accuracy: 0.2368 - loss: 4.8593 - val_accuracy: 0.1575 - val_loss: 1.3532 - learning_rate: 0.0010
Epoch 4/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 21s 20ms/step - accuracy: 0.2213 - loss: 4.7820 - val_accuracy: 0.1422 - val_loss: 1.4300 - learning_rate: 0.0010
Epoch 5/30
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.2419 - loss: 5.1780
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
1091/1091 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.2419 - loss: 5.1787 - val_accuracy: 0.1360 - val_loss: 1.4023 - learning_rate: 0.0010
Epoch 6/30
1091/1091 ━━━━━━━━━━━

/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/terlan/anaconda3/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
